# Coverage-Constrained Adaptive Readout (Colab)

Notebook 12 turns the readout-side scheduling experiments into a **control policy**.

Earlier notebooks:

```text
08 -> deterministic readout scheduling
09 -> early stopping
10 -> confidence-aware scheduling
11 -> hybrid modwheel + confidence lane coverage
```

This notebook asks:

```text
Can readout stop only after BOTH target behavior and structured coverage are satisfied?
```

Core stopping rule:

```text
stop when:
    balanced accuracy >= target
AND lane coverage >= coverage target
```

Guardrail: this notebook evaluates classical readout scheduling and stopping policies. It does **not** claim QOS improvement, quantum advantage, or model accuracy improvement.


In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)


## 1. Load 20news and train baseline classifier

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 30

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = np.array(dataset.data, dtype=object)
y = np.array(dataset.target)
target_names = dataset.target_names

texts_train, texts_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

full_pred = clf.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred)
full_balanced_accuracy = balanced_accuracy_score(y_test, full_pred)

baseline = {
    "n_train": X_train.shape[0],
    "n_test": X_test.shape[0],
    "n_features": X_train.shape[1],
    "train_time_seconds": train_time,
    "full_accuracy": full_accuracy,
    "full_balanced_accuracy": full_balanced_accuracy,
}
baseline


## 2. Confidence, lane coverage, and evaluation helpers

In [ ]:
def decision_margin_confidence(model, X):
    scores = model.decision_function(X)
    if scores.ndim == 1:
        return np.abs(scores)
    sorted_scores = np.sort(scores, axis=1)
    top = sorted_scores[:, -1]
    second = sorted_scores[:, -2]
    return top - second


confidence = decision_margin_confidence(clf, X_test)
all_idx = np.arange(X_test.shape[0])


def class_balance_l1_shift(y_subset, y_reference):
    n_classes = len(np.unique(y_reference))
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def lane_coverage_fraction(indices, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    covered = set([int(i % wheel.modulus) for i in indices if int(i % wheel.modulus) in residues])
    return len(covered) / len(residues)


def evaluate_indices(indices):
    pred = clf.predict(X_test[indices])
    yt = y_test[indices]
    return {
        "accuracy": accuracy_score(yt, pred),
        "balanced_accuracy": balanced_accuracy_score(yt, pred),
        "class_balance_l1_shift": class_balance_l1_shift(yt, y_test),
        "mean_confidence": float(np.mean(confidence[indices])),
        "median_confidence": float(np.median(confidence[indices])),
        "lane_coverage_fraction": lane_coverage_fraction(indices, "mod30"),
    }


confidence_summary = {
    "min": float(np.min(confidence)),
    "median": float(np.median(confidence)),
    "mean": float(np.mean(confidence)),
    "max": float(np.max(confidence)),
}
confidence_summary


## 3. Schedule constructors

In [ ]:
def wheel_query_indices(n_queries, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    return np.array([i for i in range(n_queries) if i % wheel.modulus in residues], dtype=int)


def random_schedule(n_keep, seed):
    rng = np.random.default_rng(seed)
    return np.array(rng.choice(all_idx, size=n_keep, replace=False), dtype=int)


def global_low_confidence_schedule(n_keep):
    return all_idx[np.argsort(confidence[all_idx])][:n_keep]


def global_high_confidence_schedule(n_keep):
    return all_idx[np.argsort(-confidence[all_idx])][:n_keep]


def mod_lanes(indices, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = list(wheel.residues)
    lanes = {r: [] for r in residues}
    for i in indices:
        r = int(i % wheel.modulus)
        if r in lanes:
            lanes[r].append(int(i))
    return lanes


def interleave_lanes(lanes):
    output = []
    keys = list(lanes.keys())
    max_len = max(len(v) for v in lanes.values() if len(v) > 0)
    for j in range(max_len):
        for k in keys:
            if j < len(lanes[k]):
                output.append(lanes[k][j])
    return np.array(output, dtype=int)


def hybrid_lane_confidence_schedule(n_keep, wheel_name="mod30", confidence_order="low"):
    base = wheel_query_indices(X_test.shape[0], wheel_name)
    lanes = mod_lanes(base, wheel_name=wheel_name)

    for r, values in lanes.items():
        arr = np.array(values, dtype=int)
        if len(arr) == 0:
            lanes[r] = []
            continue
        if confidence_order == "low":
            arr = arr[np.argsort(confidence[arr])]
        elif confidence_order == "high":
            arr = arr[np.argsort(-confidence[arr])]
        else:
            raise ValueError("confidence_order must be 'low' or 'high'")
        lanes[r] = arr.tolist()

    return interleave_lanes(lanes)[:n_keep]


def hybrid_lane_round_robin_schedule(n_keep, wheel_name="mod30"):
    base = wheel_query_indices(X_test.shape[0], wheel_name)
    lanes = mod_lanes(base, wheel_name=wheel_name)
    return interleave_lanes(lanes)[:n_keep]


mod30_idx = wheel_query_indices(X_test.shape[0], "mod30")
n_keep = len(mod30_idx)

schedules = {
    "mod30": mod30_idx,
    "global_low_conf": global_low_confidence_schedule(n_keep),
    "global_high_conf": global_high_confidence_schedule(n_keep),
    "hybrid_lane_low_conf": hybrid_lane_confidence_schedule(n_keep, "mod30", "low"),
    "hybrid_lane_high_conf": hybrid_lane_confidence_schedule(n_keep, "mod30", "high"),
    "hybrid_lane_round_robin": hybrid_lane_round_robin_schedule(n_keep, "mod30"),
}

{k: (len(v), lane_coverage_fraction(v)) for k, v in schedules.items()}


## 4. Progressive curves

In [ ]:
fractions = np.linspace(0.05, 1.0, 20)

def progressive_curve(schedule_indices, fractions, schedule_name, schedule_type, trial=-1):
    rows = []
    total_queries = X_test.shape[0]
    for frac in fractions:
        k = max(2, int(np.ceil(len(schedule_indices) * frac)))
        idx = schedule_indices[:k]
        t0 = time.perf_counter()
        metrics = evaluate_indices(idx)
        eval_time = time.perf_counter() - t0
        rows.append({
            "schedule_name": schedule_name,
            "schedule_type": schedule_type,
            "trial": trial,
            "schedule_fraction": frac,
            "n_eval": len(idx),
            "fraction_of_all_queries": len(idx) / total_queries,
            "eval_time_seconds": eval_time,
            **metrics,
        })
    return pd.DataFrame(rows)


schedule_types = {
    "mod30": "deterministic_mod30",
    "global_low_conf": "global_low_confidence",
    "global_high_conf": "global_high_confidence",
    "hybrid_lane_low_conf": "hybrid_lane_low_confidence",
    "hybrid_lane_high_conf": "hybrid_lane_high_confidence",
    "hybrid_lane_round_robin": "hybrid_lane_round_robin",
}

curve_rows = []
for name, idx in schedules.items():
    curve_rows.append(progressive_curve(idx, fractions, name, schedule_types[name], trial=-1))

for trial in range(N_RANDOM_TRIALS):
    ridx = random_schedule(n_keep, seed=RANDOM_STATE + 12000 + trial * 1009)
    curve_rows.append(progressive_curve(ridx, fractions, "random_matched", "random_matched", trial=trial))

curves_df = pd.concat(curve_rows, ignore_index=True)
curves_csv = data_dir / "12_coverage_constrained_curves.csv"
curves_df.to_csv(curves_csv, index=False)
curves_df.head()


## 5. Coverage-constrained stopping rules

In [ ]:
def stop_rule(curve_df, rule, accuracy_target=0.95, coverage_target=1.0):
    target = accuracy_target * full_balanced_accuracy

    for _, row in curve_df.iterrows():
        acc_ok = row["balanced_accuracy"] >= target
        cov_ok = row["lane_coverage_fraction"] >= coverage_target

        if rule == "accuracy_only":
            stop = acc_ok
        elif rule == "coverage_only":
            stop = cov_ok
        elif rule == "joint_accuracy_coverage":
            stop = acc_ok and cov_ok
        else:
            raise ValueError("rule must be accuracy_only, coverage_only, or joint_accuracy_coverage")

        if stop:
            return {
                "rule": rule,
                "accuracy_target_fraction": accuracy_target,
                "coverage_target": coverage_target,
                "target_balanced_accuracy": target,
                "stop_reason": "reached",
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_lane_coverage_fraction": float(row["lane_coverage_fraction"]),
                "stop_class_balance_l1_shift": float(row["class_balance_l1_shift"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
            }

    last = curve_df.iloc[-1]
    return {
        "rule": rule,
        "accuracy_target_fraction": accuracy_target,
        "coverage_target": coverage_target,
        "target_balanced_accuracy": target,
        "stop_reason": "not_reached",
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_lane_coverage_fraction": float(last["lane_coverage_fraction"]),
        "stop_class_balance_l1_shift": float(last["class_balance_l1_shift"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
    }


rules = [
    ("accuracy_only", 0.95, 0.0),
    ("coverage_only_75", 0.0, 0.75),
    ("coverage_only_100", 0.0, 1.0),
    ("joint_acc95_cov75", 0.95, 0.75),
    ("joint_acc95_cov100", 0.95, 1.0),
]

stop_rows = []
for (schedule_name, schedule_type, trial), group in curves_df.groupby(["schedule_name", "schedule_type", "trial"]):
    group = group.sort_values("schedule_fraction")
    for rule_label, acc_target, cov_target in rules:
        base_rule = "accuracy_only" if rule_label == "accuracy_only" else (
            "coverage_only" if rule_label.startswith("coverage_only") else "joint_accuracy_coverage"
        )
        result = stop_rule(group, base_rule, accuracy_target=acc_target if acc_target > 0 else 0.95, coverage_target=cov_target)
        result["rule_label"] = rule_label
        stop_rows.append({
            "schedule_name": schedule_name,
            "schedule_type": schedule_type,
            "trial": trial,
            **result,
        })

stops_df = pd.DataFrame(stop_rows)
stops_csv = data_dir / "12_coverage_constrained_stops.csv"
stops_df.to_csv(stops_csv, index=False)
stops_df.head()


## 6. Summarize vs random baseline

In [ ]:
random_summary = (
    stops_df[stops_df["schedule_type"] == "random_matched"]
    .groupby("rule_label")
    .agg(
        random_stop_fraction_mean=("stop_fraction_of_all_queries", "mean"),
        random_stop_fraction_std=("stop_fraction_of_all_queries", "std"),
        random_stop_bal_acc_mean=("stop_balanced_accuracy", "mean"),
        random_stop_bal_acc_std=("stop_balanced_accuracy", "std"),
        random_stop_lane_coverage_mean=("stop_lane_coverage_fraction", "mean"),
        random_stop_lane_coverage_std=("stop_lane_coverage_fraction", "std"),
        random_class_balance_l1_mean=("stop_class_balance_l1_shift", "mean"),
        random_class_balance_l1_std=("stop_class_balance_l1_shift", "std"),
        random_stop_n_eval_mean=("stop_n_eval", "mean"),
        random_stop_n_eval_std=("stop_n_eval", "std"),
    )
    .reset_index()
)

nonrandom = stops_df[stops_df["schedule_type"] != "random_matched"][[
    "schedule_name", "schedule_type", "rule_label", "rule",
    "accuracy_target_fraction", "coverage_target", "target_balanced_accuracy",
    "stop_reason", "stop_fraction_of_all_queries", "stop_balanced_accuracy",
    "stop_lane_coverage_fraction", "stop_class_balance_l1_shift",
    "stop_n_eval", "stop_eval_time_seconds"
]].rename(columns={
    "stop_fraction_of_all_queries": "stop_fraction",
    "stop_balanced_accuracy": "stop_bal_acc",
    "stop_lane_coverage_fraction": "stop_lane_coverage",
    "stop_class_balance_l1_shift": "stop_class_balance_l1",
})

summary_df = nonrandom.merge(random_summary, on="rule_label")
summary_df["delta_stop_fraction_vs_random"] = summary_df["stop_fraction"] - summary_df["random_stop_fraction_mean"]
summary_df["delta_stop_bal_acc_vs_random"] = summary_df["stop_bal_acc"] - summary_df["random_stop_bal_acc_mean"]
summary_df["delta_lane_coverage_vs_random"] = summary_df["stop_lane_coverage"] - summary_df["random_stop_lane_coverage_mean"]
summary_df["delta_class_balance_l1_vs_random"] = summary_df["stop_class_balance_l1"] - summary_df["random_class_balance_l1_mean"]
summary_df["delta_stop_n_eval_vs_random"] = summary_df["stop_n_eval"] - summary_df["random_stop_n_eval_mean"]

summary_csv = data_dir / "12_coverage_constrained_summary.csv"
summary_df.to_csv(summary_csv, index=False)
summary_df.head()


## 7. Figure 12a — joint stop fraction

In [ ]:
plot_rule = "joint_acc95_cov100"
plot_summary = summary_df[summary_df["rule_label"] == plot_rule].copy()
order_names = ["mod30", "global_low_conf", "hybrid_lane_low_conf", "hybrid_lane_round_robin"]
plot_summary = plot_summary.set_index("schedule_name").loc[order_names].reset_index()

labels = ["mod30", "global low-conf", "hybrid low-conf", "hybrid round-robin"]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.bar(x, plot_summary["stop_fraction"])
ax.axhline(plot_summary["random_stop_fraction_mean"].iloc[0], linestyle="--", linewidth=1, label="random mean")
ax.set_title("Joint Stop: Accuracy Target + Full Lane Coverage")
ax.set_xlabel("Schedule")
ax.set_ylabel("Fraction of all queries evaluated")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()

for i, value in enumerate(plot_summary["stop_fraction"]):
    ax.text(i, value, f"{value:.3f}", ha="center", va="bottom")

fig.tight_layout()
fig_a_svg = fig_dir / "figure_12a_joint_stop_fraction.svg"
fig_a_png = fig_dir / "figure_12a_joint_stop_fraction.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png


## 8. Figure 12b — accuracy vs lane coverage

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))

plot_schedules = [
    ("deterministic_mod30", "mod30"),
    ("global_low_confidence", "global low-conf"),
    ("hybrid_lane_low_confidence", "hybrid lanes + low-conf"),
    ("hybrid_lane_round_robin", "hybrid lane round-robin"),
]

for stype, label in plot_schedules:
    df = curves_df[curves_df["schedule_type"] == stype].sort_values("fraction_of_all_queries")
    ax.plot(df["lane_coverage_fraction"], df["balanced_accuracy"], marker="o", linewidth=2, label=label)

random_mean = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("lane_coverage_fraction")["balanced_accuracy"]
    .mean()
    .reset_index()
)
ax.plot(random_mean["lane_coverage_fraction"], random_mean["balanced_accuracy"], linewidth=2, label="random mean")

ax.axhline(0.95 * full_balanced_accuracy, linestyle=":", linewidth=1, label="95% accuracy target")
ax.axvline(1.0, linestyle="--", linewidth=1, label="100% lane coverage")
ax.set_title("Accuracy vs Mod30 Lane Coverage")
ax.set_xlabel("Lane coverage fraction")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.set_xlim(0, 1.05)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_b_svg = fig_dir / "figure_12b_accuracy_vs_lane_coverage.svg"
fig_b_png = fig_dir / "figure_12b_accuracy_vs_lane_coverage.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png


## 9. Figure 12c — class-balance shift at joint stop

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.bar(x, plot_summary["stop_class_balance_l1"])
ax.axhline(plot_summary["random_class_balance_l1_mean"].iloc[0], linestyle="--", linewidth=1, label="random mean")
ax.set_title("Class-Balance Shift at Joint Stop")
ax.set_xlabel("Schedule")
ax.set_ylabel("L1 shift vs full test distribution")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()

for i, value in enumerate(plot_summary["stop_class_balance_l1"]):
    ax.text(i, value, f"{value:.3f}", ha="center", va="bottom")

fig.tight_layout()
fig_c_svg = fig_dir / "figure_12c_class_balance_at_joint_stop.svg"
fig_c_png = fig_dir / "figure_12c_class_balance_at_joint_stop.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png


## 10. Figure 12d — stop fraction by rule

In [ ]:
rules_to_plot = ["accuracy_only", "coverage_only_100", "joint_acc95_cov100"]
compact = summary_df[
    (summary_df["schedule_name"].isin(order_names)) &
    (summary_df["rule_label"].isin(rules_to_plot))
].copy()

pivot = compact.pivot(index="schedule_name", columns="rule_label", values="stop_fraction").loc[order_names]
pivot = pivot[rules_to_plot]

fig, ax = plt.subplots(figsize=(9.5, 5.4))
im = ax.imshow(pivot.values, aspect="auto")
ax.set_title("Stop Fraction by Rule")
ax.set_xticks(np.arange(len(rules_to_plot)))
ax.set_xticklabels(["accuracy only", "coverage only", "joint"], rotation=20)
ax.set_yticks(np.arange(len(order_names)))
ax.set_yticklabels(labels)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center")

fig.colorbar(im, ax=ax, label="Fraction of all queries")
fig.tight_layout()
fig_d_svg = fig_dir / "figure_12d_stop_fraction_by_rule.svg"
fig_d_png = fig_dir / "figure_12d_stop_fraction_by_rule.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png


## 11. Figure 12e — joint stop delta vs random

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(labels, plot_summary["delta_stop_fraction_vs_random"])
ax.set_title("Joint Stop Fraction Minus Random Mean")
ax.set_xlabel("Schedule")
ax.set_ylabel("Δ stop fraction")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.grid(True, axis="y", alpha=0.35)

for i, value in enumerate(plot_summary["delta_stop_fraction_vs_random"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")

fig.tight_layout()
fig_e_svg = fig_dir / "figure_12e_joint_stop_delta_vs_random.svg"
fig_e_png = fig_dir / "figure_12e_joint_stop_delta_vs_random.png"
fig.savefig(fig_e_svg)
fig.savefig(fig_e_png, dpi=220)
plt.show()
fig_e_svg, fig_e_png


## 12. Interpretation helper

In [ ]:
print("Full balanced accuracy:", full_balanced_accuracy)
print("95% target:", 0.95 * full_balanced_accuracy)
print()
display(summary_df)

print("\nJoint stop summary:")
for _, row in plot_summary.iterrows():
    print(
        f"{row['schedule_name']}: stop_fraction={row['stop_fraction']:.3f}, "
        f"lane_coverage={row['stop_lane_coverage']:.3f}, "
        f"bal_acc={row['stop_bal_acc']:.3f}, "
        f"class_shift={row['stop_class_balance_l1']:.3f}, "
        f"random_delta={row['delta_stop_fraction_vs_random']:+.3f}"
    )

print("""
Notebook 12 interpretation:

- Accuracy-only stopping asks: when is target behavior reached?
- Coverage-only stopping asks: when has the schedule touched enough modular lanes?
- Joint stopping asks: when are BOTH conditions satisfied?

Expected tradeoff:
- joint stopping often requires more queries than accuracy-only stopping.
- the value is not better accuracy; the value is controlled stopping with coverage guarantees.

Guardrail:
This is a classical readout-control policy, not a claim of quantum advantage or QOS improvement.
""")


## 13. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/12_coverage_constrained_curves.csv",
    "data/12_coverage_constrained_stops.csv",
    "data/12_coverage_constrained_summary.csv",
    "figures/figure_12a_joint_stop_fraction.svg",
    "figures/figure_12b_accuracy_vs_lane_coverage.svg",
    "figures/figure_12c_class_balance_at_joint_stop.svg",
    "figures/figure_12d_stop_fraction_by_rule.svg",
    "figures/figure_12e_joint_stop_delta_vs_random.svg",
]:
    files.download(path)
